# Project 2 starter: Unclaimed Property at Silicone Motors

ACCTG 6155, Fall 2026. Due Sunday, October 25, 11:59 PM MT.

**First, File > Save a copy in Drive, and work in your copy.**

This notebook is optional. It gives you:

- a load cell for every table,
- a stub for each analysis step in 4.1 of the guide,
- a worked example of the reactivation curve on the same nine practice drivers the [island trainer](https://seanmccaman.com/acctg6155/2026-fall/project-2/interactive/island_trainer.html) uses, with a cell that checks your code before you run it on 8,000 real ones,
- a last section that lays out every number in 3.4 for your data support file.

The guide, the counsel memo and the data dictionary are on the [Project 2 page](https://seanmccaman.com/acctg6155/2026-fall/project-2/). The memo is the source for every rule. Nothing in this notebook tells you what the memo says, so read it before section 1.

When you are done, run every cell. This notebook, with its outputs showing, can be your data support file (section 9). The deck and the one-pager are separate files.

## 0. Load the data

The next cell loads every table from the course GitHub repository. If you work in Jupyter on your own machine and unzipped `p2_csv.zip` next to this notebook, it loads from that folder instead. Money is in integer cents. Keep it in cents until you report, because Excel, pandas and SQLite round differently.

In [ ]:
import os
import pandas as pd
from IPython.display import display

BUILD_ID = "c14c2737ec6d"   # the data build this notebook expects; the row counts below are the check
RAW = "https://raw.githubusercontent.com/sean-mccaman/acctg5150-090/main/2026-fall/project-2/data/csv/"
LOCAL = "p2_csv"          # the unzipped CSV download, if you have it

TABLES = ['drivers', 'wallet_ledger', 'charging_sessions', 'app_login_days', 'reservations', 'reservation_events', 'company_notices', 'due_diligence_letters', 'gl_wallet_control', 'run_params']
EXPECTED_ROWS = {'drivers': 8000, 'wallet_ledger': 57916, 'charging_sessions': 39556, 'app_login_days': 32135, 'reservations': 2104, 'reservation_events': 3048, 'company_notices': 12673, 'due_diligence_letters': 422, 'gl_wallet_control': 270, 'run_params': 1}
TEXT_COLUMNS = ["state_raw", "postal_raw", "country_raw", "address_line1", "city", "entered_by", "gl_account"]


def load(name):
    local = os.path.join(LOCAL, name + ".csv")
    df = pd.read_csv(local if os.path.exists(local) else RAW + name + ".csv",
                     dtype={c: str for c in TEXT_COLUMNS})
    for c in df.columns:
        if c.endswith(("_on", "_date", "_end", "_open", "_close")):
            df[c] = pd.to_datetime(df[c])
    return df


data = {t: load(t) for t in TABLES}
for t in TABLES:
    flag = "" if len(data[t]) == EXPECTED_ROWS[t] else "   <- not the expected count; reload from the Project 2 page"
    print(f"{t:24s}{len(data[t]):>8,d} rows{flag}")

drivers      = data["drivers"]
ledger       = data["wallet_ledger"]
sessions     = data["charging_sessions"]
logins       = data["app_login_days"]
reservations = data["reservations"]
res_events   = data["reservation_events"]
notices      = data["company_notices"]
letters      = data["due_diligence_letters"]
gl           = data["gl_wallet_control"]
params       = data["run_params"].iloc[0]

CUTOFF = params["cutoff_date"]
print("\nCutoff:", CUTOFF.date(), "| expected build:", BUILD_ID)


def dollars(cents):
    """For display only. Do the arithmetic in cents."""
    return f"${cents / 100:,.2f}"

In [ ]:
ledger.head()

Prefer SQL? The same tables are in one SQLite file. This cell uses your copy if one is next to the notebook, downloads it otherwise, and runs a query. After that you can use `q("...")` for the rest of the project. If you work in pandas only, you can skip it.

In [ ]:
import sqlite3
import urllib.request

DB_FILE = next((p for p in ["p2_wallet.sqlite", os.path.join("data", "p2_wallet.sqlite")] if os.path.exists(p)), None)
if DB_FILE is None:
    try:
        urllib.request.urlretrieve("https://raw.githubusercontent.com/sean-mccaman/acctg5150-090/main/2026-fall/project-2/data/p2_wallet.sqlite", "p2_wallet.sqlite")
        DB_FILE = "p2_wallet.sqlite"
    except Exception as e:
        print(f"Could not download the database ({e}). Download it from the Project 2 page and put it "
              "next to this notebook, or skip this cell and work in pandas.")

if DB_FILE:
    db = sqlite3.connect(DB_FILE)

    def q(sql):
        return pd.read_sql(sql, db)

    display(q("SELECT txn_type, COUNT(*) AS n FROM wallet_ledger GROUP BY txn_type ORDER BY n DESC"))

## 1. The rule table (4.1 step 1)

Build it from the counsel memo before you write any analysis code, and check every cell against the memo, not against an AI. Schedule A lists every state by its full name. Your table needs a row for each of the nine states where the company has drivers, keyed by the two-letter code the data uses. Add a column for each rule the memo sets by state. Rules that apply to every state go in the cell after, as plain Python values, so the code below uses them instead of retyped numbers.

The memo has a footnote that changes a number.

In [ ]:
STATES = ['AZ', 'CA', 'CO', 'DE', 'NV', 'NY', 'TX', 'UT', 'WA']   # the nine states with drivers, as the data codes them, in Schedule A's order
rules = pd.DataFrame(index=pd.Index(STATES, name="state"))

# One column per rule in the memo's schedules. For example:
# rules["wallet_period_years"] = [...]   # nine values, in the order of STATES

rules

In [ ]:
# Memo-wide rules. Fill these in from the memo.
OWNER_CONTACT_CHANNELS = set()   # source_channel values that are the owner's own doing
OWNER_CONTACT_TYPES = set()      # txn_type values that can be owner contact
FALLBACK_STATE = None            # where a balance reports when the memo's priority rule sends it elsewhere

## 2. Rebuild the balances and tie them to the GL (4.1 step 2; 3.4 items 1 to 3)

Sum the ledger per driver as of the cutoff, on the date each transaction happened (`activity_on`), with cash and promotional credit kept apart. The data dictionary lists the transaction types. Then compare your totals with `gl_wallet_control` at June 30.

One of them will not tie. The difference has a cause you can find in the data, and it is a reconciling item to explain, not an error to plug.

In [ ]:
CASH_TYPES = set()    # txn_type values that move the owner's cash
PROMO_TYPES = set()   # txn_type values that move promotional credit

# asof = ledger[ledger["activity_on"] <= CUTOFF]
# cash_by_driver = ...
# promo_by_driver = ...

gl[gl["period_end"] == CUTOFF]

## 3. The owner summary table (4.1 step 3)

Start from `drivers`, because it holds the whole population, and bring everything else in with a left join (`how="left"`), so no owner drops out. The goal is one row per owner with these columns: `driver_id`, `clean_state`, jurisdiction, cash balance, promo balance, last owner wallet transaction, last login, last letter response, last owner contact, and mailable. Some columns fill in here and the rest in section 4. Deposits run their own clocks, so build a second table the same way, one row per reservation.

Use the memo's definition of owner contact and nothing else. Two mistakes cost real money here: entries the company generated, and transactions that were later reversed. The ledger links each reversal to the row it backs out. A refund also fills in `reverses_txn_id`, pointing at the charge it corrects, so read the type: only a reversal erases the row it points to. A reply to a due diligence letter from an earlier cycle is owner contact.

In [ ]:
reversals = ledger[ledger["txn_type"] == "reversal"]
reversals[["txn_id", "driver_id", "activity_on", "amount_cents", "source_channel", "reverses_txn_id"]].head()

In [ ]:
owners = drivers[["driver_id", "state_raw"]].copy()   # one row per owner; bring everything else in with how="left"

# owners["cash_cents"] = ...             # cash balance at the cutoff, from section 2
# owners["promo_cents"] = ...            # promotional credit at the cutoff
# owners["last_wallet_txn"] = ...        # owner wallet transactions only: no company entries, nothing reversed, on or before the cutoff
# owners["last_login"] = ...
# owners["last_letter_response"] = ...   # replies to letters from earlier cycles
# clean_state, jurisdiction, last_contact and mailable come in section 4

assert owners["driver_id"].is_unique     # rerun after every join: a join that adds rows breaks it

deposits = reservations[["reservation_id", "driver_id"]].copy()   # one row per reservation, with its own clock
# deposits["deposit_cents"] = ...        # the deposit still held at the cutoff
# deposits["last_contact"] = ...         # owner-initiated events on that reservation, from res_events
assert deposits["reservation_id"].is_unique

## 4. Clean the state and resolve the jurisdiction (4.1 step 4)

`state_raw` was typed by people. Look at every spelling before you map anything. The first cell prints each value with `repr`, which shows the quotes, so a stray space that a plain print hides is visible. Put the cleaned code in a new `clean_state` column next to `state_raw`. Never overwrite the raw field, so you can sample rows and check your mapping. A join that silently drops the rows it cannot match is the most common way this analysis goes wrong, so end this section with a check that no owner is left without a jurisdiction.

Then finish the owner summary table. Whether a login counts as contact depends on the jurisdiction, so last owner contact comes after it.

In [ ]:
drivers["state_raw"].map(repr).value_counts(dropna=False)

In [ ]:
drivers["country_raw"].value_counts(dropna=False)

In [ ]:
# STATE_MAP = {...}                      # every spelling you saw, to a two-letter code
# owners["clean_state"] = ...            # a new column next to state_raw; never overwrite state_raw
# owners["jurisdiction"] = ...           # the memo's priority rule decides this, not the state column alone
# assert owners["jurisdiction"].notna().all()

# owners["last_contact"] = ...           # the latest owner contact; a login counts only where the jurisdiction's schedule says so
# owners["mailable"] = ...               # whether this cycle's letter can be mailed; the memo says when it cannot

## 5. Apply the rules (4.1 step 4; 3.4 items 4 to 7)

Add the dormancy period to the last contact date and compare the result with the cutoff. An anniversary on the cutoff itself is dormant. Months are calendar months, counted the way Excel's `EDATE` counts them, and `add_months` does that for you: August 31 plus 6 months is the last day of February. Counting days instead (365 for a year) drifts by a day around leap years, and a day is enough to move an owner across the cutoff.

In [ ]:
def add_months(dates, k):
    """Add k calendar months to a Series of dates, the way Excel's EDATE does."""
    return pd.to_datetime(dates) + pd.DateOffset(months=k)


def add_months_each(dates, months):
    """Like add_months, when each row has its own number of months (for example, a period by state)."""
    out = pd.Series(pd.NaT, index=dates.index, dtype="datetime64[ns]")
    for k in pd.unique(months.dropna()):
        rows = months == k
        out[rows] = add_months(dates[rows], int(k))
    return out


add_months(pd.Series(pd.to_datetime(["2023-08-31", "2024-02-29", "2023-06-30"])), 6)

In [ ]:
# dormant wallets: count, and reportable amount by state and in total    (3.4 items 4 and 6)
# dormant deposits: count, and reportable amount by state and in total   (3.4 items 5 and 6)
# due diligence letters: the list of owners (driver_id) who get one this cycle, and the count   (3.4 item 7)
#   The list has no bearing on your other numbers: everything is as of June 30,
#   before any of this year's letters go out.

# Checks worth running as you go: one row per owner (or per reservation) after every join,
# no wallet from an exempt state in the dormant count, and at most one letter per owner.

## 6. The reactivation curve (4.1 step 5; 3.4 item 8), on practice data first

The curve answers one question: if an owner has gone k months without a wallet transaction, how likely are they to make one in the next 24 months? You estimate it from history, one quiet spell at a time, at k = 12 and k = 36.

The [island trainer](https://seanmccaman.com/acctg6155/2026-fall/project-2/interactive/island_trainer.html) walks through every step on the same practice data this section uses: nine made-up drivers, D1 to D9. The trainer starts from their full ledger, including the rows that do not count. The table `TOY` below starts after that filter. It holds only their owner wallet transactions, one row per transaction.

**Quiet spells.** Sort each owner's transaction dates. Transactions on the same day count once. Each date starts a quiet spell that ends at the next date.

**The open spell.** The owner's last date starts a spell that is still going on June 30, 2026. That spell has not ended: you know it has lasted at least this long, not how long it will last, so it belongs in the population and can only count at the landmarks it has already reached. D7 and D8 both made their last transaction on June 20, 2026. D7 is a brand-new account with that one transaction. D8 has been a customer since 2018. Each has a quiet spell of at least 10 days that is still going, and the answer is the same for both: neither spell counts at any landmark yet.

**For k = 12 and k = 36:**

- A spell **reached k** when its k-month anniversary is on or before June 30, 2026 and the next transaction, if any, is on or after that anniversary.
- The owner **came back** when the next transaction is on or before the date 24 months after the anniversary.
- **Naive rate:** of the spells that reached k, the share where the owner came back.
- **Corrected rate:** the same, counting only spells whose 24-month window has fully ended by June 30, 2026 (the anniversary plus 24 months is on or before June 30, 2026). The calendar decides which spells count, never the outcome.

Months are calendar months, counted the way Excel's `EDATE` counts them (in pandas, `+ pd.DateOffset(months=k)`). The `add_months` function from section 5 does this.

Driver D6, at k = 12:

| Spell | Start | Next | 12-month anniversary | Reached 12? | Window ends (anniversary + 24 months) | Naive | Corrected |
|---|---|---|---|---|---|---|---|
| 1 | 2020-08-31 | 2021-03-01 | 2021-08-31 | No, the next transaction came first |  | not counted | not counted |
| 2 | 2021-03-01 | 2021-08-31 | 2022-03-01 | No, the next transaction came first |  | not counted | not counted |
| 3 | 2021-08-31 | 2024-08-31 | 2022-08-31 | Yes | 2024-08-31 | came back | came back |
| 4 | 2024-08-31 | 2025-02-12 | 2025-08-31 | No, the next transaction came first |  | not counted | not counted |
| 5 | 2025-02-12 | none yet: still going | 2026-02-12 | Yes | 2028-02-12 | did not come back (yet) | not counted: the window ends after June 30, 2026 |

Naive, D6 comes back 1 time in 2. Corrected, 1 time in 1. The difference is spell 5, the open spell. Its 24-month window ends in 2028, so the naive rate scores it as a failure before it has had its chance. Across thousands of owners, that is why the naive rate reads low.

Spell 3 is a boundary case. The next transaction lands exactly on the day the window ends, and on or before counts, so the owner came back.

Write the two functions below, then run the check. If the check disagrees with you, work the drivers by hand in the trainer first. Each one was chosen to test one rule in 3.4. D1 to D3 test the filter, which the trainer shows.

In [ ]:
# The practice data: owner wallet transactions for nine made-up drivers, after the filter.
TOY = pd.DataFrame([
    ("D1", "2021-03-02"), ("D1", "2021-03-09"), ("D1", "2023-01-20"), ("D2", "2020-09-14"),
    ("D2", "2021-02-01"), ("D3", "2021-01-15"), ("D3", "2021-02-10"), ("D4", "2019-11-04"),
    ("D4", "2019-11-04"), ("D4", "2019-12-20"), ("D4", "2023-01-05"), ("D4", "2023-01-05"),
    ("D5", "2020-05-11"), ("D5", "2021-08-16"), ("D5", "2023-03-01"), ("D5", "2024-09-30"),
    ("D6", "2020-08-31"), ("D6", "2021-03-01"), ("D6", "2021-08-31"), ("D6", "2024-08-31"),
    ("D6", "2025-02-12"), ("D7", "2026-06-20"), ("D8", "2018-04-02"), ("D8", "2019-07-15"),
    ("D8", "2020-10-01"), ("D8", "2023-01-10"), ("D8", "2024-05-06"), ("D8", "2026-06-20"),
    ("D9", "2019-02-14"), ("D9", "2022-03-01"), ("D9", "2024-12-02"),
], columns=["owner", "activity_on"])
TOY["activity_on"] = pd.to_datetime(TOY["activity_on"])
TOY_CUTOFF = pd.Timestamp("2026-06-30")   # the practice data also ends on this date
TOY

In [ ]:
def make_gaps(txns):
    """
    txns: one row per owner wallet transaction, with columns owner and activity_on.

    Return one row per quiet spell, with columns owner, start, next.
    Transactions on the same day count once. The open spell after an owner's
    last transaction has next = NaT.
    """
    return None   # your code


def curve_at(gaps, k, cutoff, window=24):
    """
    Return a dict with the two rates for landmark k: {"naive": ..., "corrected": ...}
    Use add_months for every anniversary.
    """
    return None   # your code

In [ ]:
# ===== RUN THIS to check your two functions on the practice data. You do not edit this cell. =====
from numbers import Real

EXPECTED = {'n_gaps': 29,
 'n_open': 9,
 'per_owner': {'D1': 3, 'D2': 2, 'D3': 2, 'D4': 3, 'D5': 4, 'D6': 5, 'D7': 1, 'D8': 6, 'D9': 3},
 'same_day_owner': 'D4',
 'curve': {12: {'naive': [11, 20], 'corrected': [10, 16]},
           36: {'naive': [3, 7], 'corrected': [2, 4]}},
 'open_spell_mistakes': {12: {'dropped': 0.8333333333333334, 'closed_as_return': 0.875},
                         36: {'dropped': 1.0, 'closed_as_return': 1.0}}}
OPEN_SPELL_HINTS = {
    "dropped": "That is the rate you get by dropping the open spells. They have not ended, and they belong in the population.",
    "closed_as_return": "That is the rate you get by counting an open spell as a return. The data stops at the cutoff. The spell does not.",
    "both": "That is the rate you get by dropping the open spells or by counting them as returns. An open spell has not ended, and it stays in the population."
}


def check_curve():
    results = []

    def record(name, ok, hint=""):
        results.append((name, "done" if ok else "not yet", "" if ok else hint))

    gaps = None
    try:
        gaps = make_gaps(TOY.copy())
        if gaps is None:
            record("make_gaps", False, "returns None. Write it first.")
    except Exception as e:
        record("make_gaps runs", False, f"it raised {type(e).__name__}: {e}")
    if gaps is not None:
        try:
            cols_ok = {"owner", "start", "next"} <= set(gaps.columns)
            record("make_gaps has the three columns", cols_ok, "need owner, start, next")
            o = EXPECTED["same_day_owner"]
            starts = pd.to_datetime(gaps.loc[gaps["owner"] == o, "start"])
            extra = int(starts.duplicated().sum())
            record("same-day transactions count once", len(starts) > 0 and extra == 0,
                   f"{o} has no spells" if len(starts) == 0 else
                   f"{o} has {len(starts)} spells, and {extra} of them start on a day another spell starts. "
                   "Transactions on the same day count once")
            n_open = int(gaps["next"].isna().sum())
            record("one spell per transaction date, the last one open",
                   len(gaps) == EXPECTED["n_gaps"] and n_open == EXPECTED["n_open"],
                   f"expected {EXPECTED['n_gaps']} spells, {EXPECTED['n_open']} of them open; "
                   f"you have {len(gaps)}, {n_open} open")
        except Exception as e:
            record("make_gaps output", False, f"checking it raised {type(e).__name__}: {e}")

    for k, want in EXPECTED["curve"].items():
        if gaps is None:
            record(f"curve_at k={k}", False, "waiting on make_gaps")
            continue
        try:
            got = curve_at(gaps, k, TOY_CUTOFF)
        except Exception as e:
            record(f"curve_at k={k}", False, f"it raised {type(e).__name__}: {e}")
            continue
        if not isinstance(got, dict):
            record(f"curve_at k={k}", False, "should return a dict with naive and corrected")
            continue
        for name, (came_back, spells) in want.items():
            have = got.get(name)
            target = came_back / spells
            is_rate = isinstance(have, Real) and not isinstance(have, bool)
            ok = is_rate and abs(have - target) < 1e-9
            shown = f"{have:.4f}" if is_rate else repr(have)
            hint = f"expected {came_back} of {spells} spells = {target:.4f}, got {shown}"
            if not ok and name == "corrected" and is_rate:
                hits = [m for m, rate in EXPECTED["open_spell_mistakes"][k].items() if abs(have - rate) < 1e-9]
                if hits:
                    hint += ". " + OPEN_SPELL_HINTS[hits[0] if len(hits) == 1 else "both"]
            record(f"curve_at k={k} {name}", ok, hint)

    done = sum(r[1] == "done" for r in results)
    print(f"{done} of {len(results)} checks done\n")
    for name, status, hint in results:
        print(f"  {status:8s} {name}" + (f"   ({hint})" if hint else ""))
    if done == len(results):
        print("\nYour functions match the rules in 3.4 on the practice data. Now build the real spells.")


check_curve()

### Now the real curve

Build the real quiet spells from **owner wallet transactions**, as 3.4 item 8 defines them: the qualifying loads, charges, refunds and promotional redemptions the owner initiated (the memo says which channels and types count) that were not reversed, dated on the day they happened, on or before the cutoff. A login or a letter response is contact for the dormancy question, but it is not a return here, so this is not the same set of rows you used in section 4.

Each spell counts once. There is no weighting by balance.

In [ ]:
LANDMARKS = [12, 36]

# 1. The transactions: owner wallet transactions as 3.4 item 8 defines them.
# real_txns = ledger[...][["driver_id", "activity_on"]].rename(columns={"driver_id": "owner"})

# 2. The spells and the rates.
# real_gaps = make_gaps(real_txns)
# curve = pd.DataFrame([{"k": k, **curve_at(real_gaps, k, CUTOFF)} for k in LANDMARKS]).set_index("k")
# curve.round(4)

In [ ]:
# import matplotlib.pyplot as plt
# ax = curve[["naive", "corrected"]].plot(kind="bar", rot=0, figsize=(7, 4.5))
# ax.set_xlabel("Months since the owner's last wallet transaction (k)")
# ax.set_ylabel("Share that transact within the next 24 months")
# ax.set_title("A title that states the finding, not the topic")

## 7. Breakage (4.1 step 6; 3.4 item 9)

- **The base.** Take the wallet cash at June 30, 2026, on the date each transaction happened, held by owners in the states that exempt stored value, after you resolve each owner's jurisdiction under the memo's rules. Keep only owners whose last owner wallet transaction was 36 months or more before June 30, 2026 (its 36-month anniversary is on or before the cutoff).
- **The estimate.** Multiply that base by one minus the corrected 36-month rate from section 6, the rate across all owners, unrounded. Round only the final dollars.

This is breakage, not escheatment. In the reporting states dormant cash goes to the state, and only in the exempt states does the company keep it. Write down what the estimate assumes while you build it. The auditors will ask what would make it wrong.

In [ ]:
# breakage_base_cents = ...
# rate_36 = curve.loc[36, "corrected"]   # unrounded
# breakage_cents = ...

## 8. The numbers

Fill in every value from your work above, by variable, not by typing a number in. Dollar values go in as cents and print as dollars to the cent. Rates go in as decimals and print to four places. Items 4 and 5 also ask for tables: the reportable amounts by state. Put the by-state amounts in dollars (divide the cents by 100 as the last step) so they read in the units 3.4 asks for. Item 7 asks for the list itself: put the driver_ids that get a letter in `letter_list`, and the count in `numbers`.

Then run the last cell. It prints everything in 3.4 order and lists anything still missing.

In [ ]:
numbers = {
    "1   Wallet cash liability at June 30":                  None,   # cents
    "2   GL 2110 minus your rebuilt cash liability":         None,   # cents
    "2   Transactions that explain the difference":          None,   # count
    "2   Cause of the difference":                           None,   # one sentence
    "3   Promotional credit at June 30":                     None,   # cents
    "3   Ties to GL 2115":                                   None,   # "yes" or "no"
    "4   Dormant wallets":                                   None,   # count
    "4   Reportable wallet cash, total":                     None,   # cents
    "5   Dormant deposits":                                  None,   # count
    "5   Reportable deposits, total":                        None,   # cents
    "6   Reportable to Delaware, both property types":       None,   # cents
    "6   Why each group reports to Delaware":                None,   # a sentence per group
    "7   Due diligence letters this cycle, count":           None,   # count
    "8   Naive rate, 12 months":                             None,   # rate
    "8   Corrected rate, 12 months":                         None,   # rate
    "8   Naive rate, 36 months":                             None,   # rate
    "8   Corrected rate, 36 months":                         None,   # rate
    "9   Breakage estimate":                                 None,   # cents
}
UNITS = {"1": "cents", "2   GL": "cents", "3   Promotional": "cents", "4   Reportable": "cents",
         "5   Reportable": "cents", "6   Reportable": "cents", "8": "rate", "9": "cents"}
LETTER_COUNT = "7   Due diligence letters this cycle, count"

letter_list = None        # list: the driver_ids that get a letter this cycle          (item 7)
wallet_by_state = None    # table: state, dormant wallets, reportable dollars           (item 4)
deposit_by_state = None   # table: state, dormant deposits, reportable dollars          (item 5)

In [ ]:
import textwrap
from numbers import Integral, Real
from IPython.display import display

print(f"Project 2 numbers, data build {BUILD_ID}\n")
missing = []
for label, value in numbers.items():
    unit = next((u for prefix, u in UNITS.items() if label.startswith(prefix)), None)
    if value is None:
        missing.append(label)
        shown = "MISSING"
    elif unit == "cents" and isinstance(value, Real):
        shown = dollars(value)
    elif unit == "rate" and isinstance(value, Real):
        shown = f"{value:.4f}"
    elif isinstance(value, Integral) and not isinstance(value, bool):
        shown = f"{int(value):,}"
    else:
        shown = str(value)
    print(f"{label:52s} {shown}")

print("\n7   Due diligence letter list (driver_id)")
if letter_list is None:
    missing.append("7   Due diligence letter list")
    print("MISSING")
else:
    ids = sorted(int(i) for i in letter_list)
    note = ""
    if numbers.get(LETTER_COUNT) is not None and int(numbers[LETTER_COUNT]) != len(ids):
        note += "   <- not the count above"
    if len(set(ids)) != len(ids):
        note += "   <- the list repeats an owner; a letter goes to each owner once"
    print(f"{len(ids):,} owners{note}")
    print(textwrap.fill(", ".join(str(i) for i in ids), width=100))

for name, table in [("4   Reportable wallet cash by state", wallet_by_state),
                    ("5   Reportable deposits by state", deposit_by_state)]:
    print(f"\n{name}")
    if table is None:
        missing.append(name)
        print("MISSING")
    else:
        display(table)

print("\nStill missing:" if missing else "\nEvery 3.4 number is present.")
for label in missing:
    print("  ", label.strip())

## 9. Export

Run every cell from the top (in Colab, Runtime > Run all) and check that the outputs are showing. This notebook, with its outputs, is your data support file. Download the notebook itself (in Colab, File > Download > Download .ipynb), or print it to PDF (File > Print, then Save as PDF). Upload it to the data support file question in the Project 2 quiz, with your deck and your one-pager.